# 02 - Foods Data And Chunking

Notebook này khám phá dữ liệu foods đã curate và cách pipeline chia nội dung thành đoạn (chunk) cho RAG. Notebook dùng lại các module của backend ở Phase 2, không chép lại logic chia đoạn.

Chạy từng khối mã từ repo root hoặc từ `notebooks/`; khối đầu tiên tự tìm đường dẫn `backend/` theo thư mục đang làm việc. Notebook chỉ đọc dữ liệu local, không gọi dịch vụ ngoài.

In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
print(f"backend on path: {sys.path[0]}")

In [ ]:
from ingestion.chunking.markdown_chunker import _discover_markdown_files

root, files = _discover_markdown_files()
print(f"knowledge base root: {root}")
print(f"markdown files indexed: {len(files)}")
print()
for path in files[:5]:
    print(path.relative_to(root))
print("...")

## Đoạn nội dung (chunk) là gì và giới hạn 400 ký tự

Một đoạn nội dung là đơn vị nhỏ, đứng độc lập, mà pipeline sẽ đưa vào index và retrieval sau này. Mỗi mục H2 (`##`) của file đã curate có thể tạo ra **một hoặc nhiều đoạn**; mục `## Nguồn dữ liệu` bị loại vì không phải nội dung trả lời.

Phần nội dung thường của mỗi đoạn giới hạn ở **400 ký tự**. Con số này nhỏ hơn trước đây (1.500) để mỗi đoạn tập trung đúng một ý, dễ khớp câu hỏi của người dùng hơn.

Thứ tự ưu tiên khi chia một mục quá dài:

1. Ranh giới đoạn văn (dòng trống).
2. Cuối câu (sau dấu chấm, chấm than, chấm hỏi).
3. Giữa các mục danh sách.
4. Khoảng trắng gần nhất trước giới hạn, nếu một câu vẫn quá dài.

Với danh sách, một mục gồm dòng bắt đầu bằng dấu danh sách và các dòng xuống hàng của chính mục đó. Các dòng này luôn đi cùng nhau; pipeline chỉ chia **giữa các mục**, và chỉ chia bên trong một mục khi riêng mục đó dài hơn 400 ký tự.

Hệ quả: không bao giờ cắt giữa từ, và hai đoạn liên tiếp không chồng lặp ký tự.

In [ ]:
from collections import Counter
from statistics import median

from ingestion.chunking.markdown_chunker import chunk_foods_markdown

chunks = chunk_foods_markdown()
contents = [len(c["text"].split("\n", 1)[1]) for c in chunks]
print(f"tổng số đoạn: {len(chunks)}")
print("theo nhóm:", dict(Counter(c["metadata"]["subcategory"] for c in chunks)))
print("số file được xử lý:", len({c["metadata"]["source"] for c in chunks}))
print(
    "độ dài phần nội dung: "
    f"trung bình {sum(contents) / len(contents):.1f}, "
    f"trung vị {median(contents)}, lớn nhất {max(contents)}"
)

## Bảng Markdown và ngoại lệ 400 ký tự

Bảng Markdown được viết bằng dấu `|` để phân cách cột:

```text
| Món | Giá |
|---|---:|
| Bún bò | 40.000 VNĐ |
```

Hàng đầu tiên là tiêu đề cột, hàng thứ hai (`---`) là đường kẻ phân cách, các hàng tiếp theo là dữ liệu. Nếu tách rời bảng, hàng và cột sẽ vỡ, người đọc không còn hiểu được nội dung.

Vì vậy bảng là **ngoại lệ có chủ ý**: một bảng luôn được giữ nguyên cả khối, kể cả khi dài hơn 400 ký tự.

## Nhãn ngữ cảnh

Mỗi đoạn bắt đầu bằng một dòng nhãn dạng `Tên tài liệu — nhãn ngắn`, ví dụ:

```text
ANH KAFE tại Huế — địa chỉ
```

Nhãn được tạo bằng quy tắc cố định từ tiêu đề tài liệu và tên mục, không gọi mô hình và không thêm dữ liệu mới:

- nhóm restaurants/cafes: `giới thiệu`, `địa chỉ`, `giờ hoạt động`, `mức giá`, `menu`, `trải nghiệm`;
- nhóm local specialties: `giới thiệu`, `thành phần`, `cách làm`, `nguồn gốc`, `địa điểm`;
- file `food-guides.md`: chủ đề ngắn của mục như `ăn sáng`, `ăn tối`, `tour 1 ngày`;
- mục chung như `Thông tin`: dùng nhãn cụ thể khi đoạn chỉ có đúng một chủ đề, còn lại dùng `thông tin quán`.

Nhãn **không tính** vào giới hạn 400 ký tự; giới hạn chỉ áp dụng cho phần nội dung thật.

### Ví dụ 1 - một đoạn văn

Đoạn đầu tiên thuộc loại đoạn văn: nội dung là câu văn thường, dưới 400 ký tự, có nhãn `giới thiệu`. Dòng in ra là dữ liệu đoạn thật từ pipeline; phần hiển thị Markdown bên dưới chỉ là bản xem trước cho dễ đọc.

In [ ]:
from IPython.display import Markdown, display

para = next(
    c for c in chunks
    if c["metadata"]["section"] == "Tóm tắt"
    and "|" not in c["text"].split("\n", 1)[1]
)
label_line, content = para["text"].split("\n", 1)
print("nhãn:", label_line)
print("độ dài phần nội dung:", len(content))
print("mã đoạn:", para["metadata"]["chunk_id"])
display(Markdown(para["text"]))

### Ví dụ 2 - một bảng Markdown

Đoạn này chứa một bảng menu. Bảng được giữ nguyên cả khối: dòng `| ... |` là tiêu đề cột, dòng `|---|` là đường kẻ phân cách. Bản xem trước dưới đây hiển thị đúng hàng và cột nhờ cú pháp Markdown; nội dung thật vẫn nằm trong trường `text` của đoạn.

In [ ]:
table = next(c for c in chunks if "|" in c["text"].split("\n", 1)[1])
label_line, content = table["text"].split("\n", 1)
print("nhãn:", label_line)
print("độ dài phần nội dung:", len(content), "(bảng được phép vượt 400)")
print("mã đoạn:", table["metadata"]["chunk_id"])
display(Markdown(table["text"]))

### Ví dụ 3 - một đoạn trong food-guides.md

File `food-guides.md` tổng hợp gợi ý theo nhu cầu du khách. Một mục của file này có thể tạo ra **một hoặc nhiều đoạn**; mỗi đoạn mang nhãn là chủ đề ngắn như `ăn sáng`, `món chay` hay `tour 1 ngày`.

In [ ]:
guide = next(c for c in chunks if c["metadata"]["subcategory"] == "guide")
label_line, content = guide["text"].split("\n", 1)
print("nhãn:", label_line)
print("độ dài phần nội dung:", len(content))
print("mã đoạn:", guide["metadata"]["chunk_id"])
display(Markdown(guide["text"]))

## Kiểm tra gate

Chạy các kiểm tra cốt lõi trên toàn bộ đoạn: mã đoạn unique, nội dung không rỗng, không có đường dẫn tuyệt đối, mục `Nguồn dữ liệu` và dòng ảnh đã bị loại, đoạn thường không vượt 400 ký tự và mỗi đoạn có đúng một nhãn. Phép nhận diện bảng được dùng lại trực tiếp từ backend.

In [ ]:
from ingestion.helpers.split_text import _is_table, _split_blocks


def _chunk_has_table(content):
    return any(_is_table(block) for block in _split_blocks(content))


ids = [c["metadata"]["chunk_id"] for c in chunks]
assert len(ids) == len(set(ids)), "mã đoạn phải unique"
assert all(c["text"].strip() for c in chunks), "nội dung không được rỗng"
assert all(not c["metadata"]["source"].startswith("/") for c in chunks), "không có đường dẫn tuyệt đối"
assert all(c["metadata"]["section"] != "Nguồn dữ liệu" for c in chunks), "đã loại mục Nguồn dữ liệu"
assert all("![" not in c["text"] for c in chunks), "đã loại dòng ảnh"
assert all(
    c["text"].split("\n", 1)[0].startswith(c["metadata"]["title"] + " — ")
    for c in chunks
), "mỗi đoạn có đúng một nhãn"

normal_over = sum(
    1 for c in chunks
    if not _chunk_has_table(c["text"].split("\n", 1)[1])
    and len(c["text"].split("\n", 1)[1]) > 400
)
table_over = sum(
    1 for c in chunks
    if _chunk_has_table(c["text"].split("\n", 1)[1])
    and len(c["text"].split("\n", 1)[1]) > 400
)
assert normal_over == 0, "đoạn thường không được vượt 400 ký tự"
print("kiểm tra gate đạt:", len(chunks), "đoạn")
print("đoạn thường vượt 400:", normal_over)
print("bảng vượt 400:", table_over)

## Các trường hợp biên đã xử lý

- Dòng chỉ chứa ảnh bị bỏ khỏi nội dung đoạn (không mang thông tin trả lời).
- Mục `## Nguồn dữ liệu` không thành đoạn; nguồn vẫn truy vết được qua trường `source` trong metadata.
- Bảng Markdown luôn giữ nguyên khối, kể cả khi dài hơn 400 ký tự.
- Danh sách ưu tiên chia giữa các mục; các dòng xuống hàng thụt lề thuộc cùng mục và đi cùng nhau; một mục dài hơn 400 ký tự vẫn có thể bị chia đôi.
- Một câu dài không có dấu câu sẽ ngắt tại khoảng trắng gần nhất trước giới hạn.
- Tiêu đề phụ H3 vẫn nằm trong mục H2 của nó.

Mã đoạn ổn định với cùng dữ liệu đầu vào: chạy lại notebook sẽ cho cùng danh sách mã đoạn. Không có lời gọi dịch vụ ngoài nào trong notebook này.